<a href="https://colab.research.google.com/github/Rneupane0056/MADS-Course-Work/blob/main/DATA715_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# Cell 1: Install Java and PySpark in Google Colab
# =========================================================
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark pyspark

In [ ]:
# =========================================================
# Cell 2: Set environment variables
# =========================================================
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

import findspark
findspark.init()

In [ ]:
# =========================================================
# Cell 3: Start SparkSession
# =========================================================
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDD and MapReduce Examples") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark version:", spark.version)
print("SparkContext created successfully")

Spark version: 3.5.1
SparkContext created successfully


# =========================================================
# Cell 4: Introduction to RDDs
# =========================================================
print("""
RDD = Resilient Distributed Dataset

An RDD is a distributed collection of data.
It is one of the core low-level data structures in Spark.

Common RDD operations:
- parallelize()   -> create an RDD from a Python collection
- map()           -> transform each item
- filter()        -> keep matching items
- flatMap()       -> split one item into many
- reduceByKey()   -> combine values for the same key
- collect()       -> bring results back to the driver
- count()         -> count records
- take(n)         -> show first n records
""")

In [ ]:
# =========================================================
# Cell 5: Create a simple RDD
# =========================================================
numbers = [1, 2, 3, 4, 5, 6]
rdd_numbers = sc.parallelize(numbers)

print("RDD contents:", rdd_numbers.collect())
print("Number of elements:", rdd_numbers.count())

RDD contents: [1, 2, 3, 4, 5, 6]
Number of elements: 6


In [ ]:
# =========================================================
# Cell 6: Example of map()
# =========================================================
# Multiply every number by 2
doubled_rdd = rdd_numbers.map(lambda x: x * 2)

print("Original numbers:", rdd_numbers.collect())
print("Doubled numbers:", doubled_rdd.collect())

Original numbers: [1, 2, 3, 4, 5, 6]
Doubled numbers: [2, 4, 6, 8, 10, 12]


In [ ]:
# =========================================================
# Cell 7: Example of filter()
# =========================================================
# Keep only even numbers
even_rdd = rdd_numbers.filter(lambda x: x % 2 == 0)

print("Even numbers:", even_rdd.collect())

Even numbers: [2, 4, 6]


In [ ]:
# =========================================================
# Cell 8: Example of flatMap()
# =========================================================
sentences = [
    "spark is fast",
    "spark is scalable",
    "rdd is powerful"
]

rdd_sentences = sc.parallelize(sentences)

words_rdd = rdd_sentences.flatMap(lambda line: line.split(" "))

print("Sentences:", rdd_sentences.collect())
print("Words:", words_rdd.collect())

Sentences: ['spark is fast', 'spark is scalable', 'rdd is powerful']
Words: ['spark', 'is', 'fast', 'spark', 'is', 'scalable', 'rdd', 'is', 'powerful']


In [ ]:
# =========================================================
# Cell 9: Basic reduce() example
# =========================================================
# Add all numbers together
total = rdd_numbers.reduce(lambda a, b: a + b)

print("Numbers:", rdd_numbers.collect())
print("Sum:", total)

Numbers: [1, 2, 3, 4, 5, 6]
Sum: 21


# =========================================================
# Cell 10: What is MapReduce?
# =========================================================
print("""
MapReduce is a programming model with two main ideas:

1. Map
   Transform input data into key-value pairs

2. Reduce
   Aggregate values that share the same key

Classic example:
Input text:
    "cat dog cat bird dog cat"

Map step:
    (cat, 1), (dog, 1), (cat, 1), (bird, 1), ...

Reduce step:
    (cat, 3), (dog, 2), (bird, 1)

In Spark, reduceByKey() is a common way to do this.
""")

In [ ]:
# =========================================================
# Cell 11: Word Count MapReduce Example
# =========================================================
text_data = [
    "cat dog cat bird",
    "dog cat fish",
    "bird fish cat"
]

rdd_text = sc.parallelize(text_data)

word_counts = (
    rdd_text
    .flatMap(lambda line: line.split(" "))      # Map: split into words
    .map(lambda word: (word, 1))                # Map: create (word, 1)
    .reduceByKey(lambda a, b: a + b)            # Reduce: sum counts
)

print("Input text:")
for line in text_data:
    print("-", line)

print("\nWord counts:")
for item in word_counts.collect():
    print(item)

Input text:
- cat dog cat bird
- dog cat fish
- bird fish cat

Word counts:
('dog', 2)
('bird', 2)
('cat', 4)
('fish', 2)


In [ ]:
# =========================================================
# Cell 12: Show each step separately
# =========================================================
words = rdd_text.flatMap(lambda line: line.split(" "))
pairs = words.map(lambda word: (word, 1))
counts = pairs.reduceByKey(lambda a, b: a + b)

print("Step 1 - flatMap words:")
print(words.collect())

print("\nStep 2 - map to key-value pairs:")
print(pairs.collect())

print("\nStep 3 - reduceByKey results:")
print(counts.collect())

Step 1 - flatMap words:
['cat', 'dog', 'cat', 'bird', 'dog', 'cat', 'fish', 'bird', 'fish', 'cat']

Step 2 - map to key-value pairs:
[('cat', 1), ('dog', 1), ('cat', 1), ('bird', 1), ('dog', 1), ('cat', 1), ('fish', 1), ('bird', 1), ('fish', 1), ('cat', 1)]

Step 3 - reduceByKey results:
[('dog', 2), ('bird', 2), ('cat', 4), ('fish', 2)]


In [ ]:
# =========================================================
# Cell 13: Another MapReduce example with student scores
# =========================================================
scores = [
    ("Alice", 85),
    ("Bob", 90),
    ("Alice", 95),
    ("Bob", 88),
    ("Charlie", 91),
    ("Alice", 78)
]

rdd_scores = sc.parallelize(scores)

# Sum scores by student
score_totals = rdd_scores.reduceByKey(lambda a, b: a + b)

print("Original score records:")
print(rdd_scores.collect())

print("\nTotal scores by student:")
print(score_totals.collect())

Original score records:
[('Alice', 85), ('Bob', 90), ('Alice', 95), ('Bob', 88), ('Charlie', 91), ('Alice', 78)]

Total scores by student:
[('Alice', 258), ('Bob', 178), ('Charlie', 91)]


In [ ]:
# =========================================================
# Cell 14: Average score by student
# =========================================================
# Step 1: Convert each (student, score) to (student, (score, 1))
score_pairs = rdd_scores.map(lambda x: (x[0], (x[1], 1)))

# Step 2: Sum the scores and counts
score_sums_counts = score_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# Step 3: Compute average
score_averages = score_sums_counts.mapValues(lambda x: x[0] / x[1])

print("Sum and count by student:")
print(score_sums_counts.collect())

print("\nAverage score by student:")
print(score_averages.collect())

Sum and count by student:
[('Alice', (258, 3)), ('Bob', (178, 2)), ('Charlie', (91, 1))]

Average score by student:
[('Alice', 86.0), ('Bob', 89.0), ('Charlie', 91.0)]


In [ ]:
# =========================================================
# Cell 15: Example using key-value pairs for categories
# =========================================================
sales_data = [
    ("Books", 20),
    ("Books", 35),
    ("Games", 50),
    ("Games", 40),
    ("Electronics", 100),
    ("Books", 15)
]

rdd_sales = sc.parallelize(sales_data)

sales_totals = rdd_sales.reduceByKey(lambda a, b: a + b)

print("Sales totals by category:")
print(sales_totals.collect())

Sales totals by category:
[('Games', 90), ('Books', 70), ('Electronics', 100)]


In [ ]:
# =========================================================
# Cell 16: Sorting results
# =========================================================
sorted_word_counts = word_counts.sortBy(lambda x: x[1], ascending=False)

print("Word counts sorted from highest to lowest:")
print(sorted_word_counts.collect())

Word counts sorted from highest to lowest:
[('cat', 4), ('dog', 2), ('bird', 2), ('fish', 2)]


# =========================================================
# Cell 17: Practice Exercise 1
# =========================================================
print("""
Practice Exercise 1:
Create an RDD from the numbers 1 through 10.
Then:
1. Use map() to square each number
2. Use filter() to keep only numbers greater than 20
3. Display the final results
""")

In [ ]:
# =========================================================
# Cell 18: Practice Exercise 1 Solution
# =========================================================
practice_rdd = sc.parallelize(range(1, 11))
result = practice_rdd.map(lambda x: x ** 2).filter(lambda x: x > 20)

print("Squared numbers greater than 20:")
print(result.collect())

Squared numbers greater than 20:
[25, 36, 49, 64, 81, 100]


# =========================================================
# Cell 19: Practice Exercise 2
# =========================================================
print("""
Practice Exercise 2:
Given the following lines of text, perform a word count.

Lines:
- "data science is fun"
- "big data is big"
- "science uses data"

Instructions:
1. Create an RDD
2. Split each line into words
3. Convert each word to a (word, 1) pair
4. Count each word with reduceByKey()
""")

In [ ]:
# =========================================================
# Cell 20: Practice Exercise 2 Solution
# =========================================================
practice_text = [
    "data science is fun",
    "big data is big",
    "science uses data"
]

practice_text_rdd = sc.parallelize(practice_text)

practice_word_count = (
    practice_text_rdd
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)

print("Practice word count results:")
print(practice_word_count.collect())

Practice word count results:
[('science', 2), ('fun', 1), ('big', 2), ('uses', 1), ('data', 3), ('is', 2)]


# =========================================================
# Cell 21: Challenge Exercise
# =========================================================
print("""
Challenge Exercise:
Given a list of product purchases:
("Pen", 2), ("Notebook", 3), ("Pen", 4), ("Pencil", 5), ("Notebook", 2)

Tasks:
1. Create an RDD
2. Find the total quantity sold for each product
3. Sort results by total quantity descending
""")

In [ ]:
# =========================================================
# Cell 22: Challenge Exercise Solution
# =========================================================
purchases = [
    ("Pen", 2),
    ("Notebook", 3),
    ("Pen", 4),
    ("Pencil", 5),
    ("Notebook", 2)
]

rdd_purchases = sc.parallelize(purchases)

purchase_totals = rdd_purchases.reduceByKey(lambda a, b: a + b)
purchase_totals_sorted = purchase_totals.sortBy(lambda x: x[1], ascending=False)

print("Total quantity sold by product:")
print(purchase_totals.collect())

print("\nSorted totals:")
print(purchase_totals_sorted.collect())

Total quantity sold by product:
[('Pen', 6), ('Notebook', 5), ('Pencil', 5)]

Sorted totals:
[('Pen', 6), ('Notebook', 5), ('Pencil', 5)]


# =========================================================
# Cell 23: Key Takeaways
# =========================================================
print("""
Key Takeaways:

1. RDDs are Spark's low-level distributed data structure.
2. map() transforms each record independently.
3. flatMap() can turn one record into many.
4. reduce() combines all values into one result.
5. reduceByKey() is used for MapReduce-style aggregation by key.
6. Word count is the classic example of MapReduce in Spark.
7. RDDs help students understand distributed processing before moving to DataFrames.
""")

In [ ]:
# =========================================================
# Cell 24: Stop Spark when done
# =========================================================
spark.stop()
print("Spark session stopped.")